# Banking Intent Classifier

Predicts which of 77 banking intents a customer message belongs to, using the BANKING77 dataset.

## Contents

1. Problem definition
2. Setup

## 1. Problem definition

### 1.1 The problem

Bank customers often contact support with short, free-text messages. Before a chatbot can help, it has to work out what the customer wants: its **intent**.

This project builds a classifier that takes one customer message and predicts one of 77 banking intents.

| Example customer message | Intent |
|---|---|
| "My card hasn't arrived yet" | `card_arrival` |
| "I think someone has been using my card" | `compromised_card` |
| "Why was I charged twice for the same purchase?" | `transaction_charged_twice` |

This is **single-label, multi-class text classification**.

### 1.2 Users and their needs

| User | Need |
|---|---|
| **Bank customers** | Reach the right help quickly, without repeating themselves or being sent to the wrong place |
| **Support teams** | Have routine requests handled automatically, and receive the cases that need a person |
| **Chatbot developers** | A model that is accurate, fast, and clear about when it is unsure |

### 1.3 Constraints and challenges

- **Many classes:** 77 intents is far harder than a typical sentiment task with two or three labels.
- **Very similar intents:** several intents share most of their vocabulary and differ only in meaning, as shown below.
- **Short messages:** a message gives little context, and the chatbot has no conversation history to draw on.
- **Limited data:** about 13,000 labelled messages across 77 intents, so each intent has relatively few examples.
- **Informal text:** real customers make typos and use casual wording.
- **Off-topic messages:** a real chatbot receives messages outside its 77 intents, but BANKING77 has no examples of these.
- **Response time:** a chatbot has to reply quickly, so model size and speed matter as well as accuracy.

| Similar intents | The difference |
|---|---|
| `lost_or_stolen_card` / `compromised_card` | The customer no longer has the card, or the card's details have been misused |
| `card_arrival` / `card_delivery_estimate` | An expected card is missing, or the customer wants to know how long delivery takes |
| `pending_transfer` / `transfer_not_received_by_recipient` | The transfer has not completed, or it appears sent but the recipient does not have it |

### 1.4 The cost of errors

Not all mistakes are equally serious, so overall accuracy alone is not enough.

| Severity | Example | Consequence |
|---|---|---|
| **High** | `lost_or_stolen_card` predicted as `card_not_working` | The card stays active while the customer gets troubleshooting steps, so there is a risk of financial loss |
| **Medium** | `transaction_charged_twice` predicted as `request_refund` | The customer gets the wrong process and has to contact support again |
| **Low** | `card_arrival` predicted as `card_delivery_estimate` | The answer is closely related, so it is a minor inconvenience |

For this reason, the system should say when it is unsure instead of always giving an answer, and the evaluation looks at results for each intent as well as overall scores.

### 1.5 Scope

**In scope**

- English messages about the 77 BANKING77 intents
- Predicting the most likely intents, with confidence scores
- Flagging messages that are off-topic or uncertain

**Out of scope**

- Carrying out actions, such as blocking a card or sending money
- Writing full chatbot replies
- Using conversation history
- Messages containing more than one request, since each example in the dataset has one label

This is a course project trained on a public dataset. It is not a production banking system.

### 1.6 Success criteria

These targets are set by this project, and later sections report whether each one was met.

| Requirement | Measure | Target |
|---|---|---|
| Correct classification | Macro F1 on the test set | At least 0.90 |
| Reported performance | Accuracy, precision, recall and F1 | All reported on the test set |
| Better than a simple approach | Macro F1 compared with the TF-IDF baseline | Higher than the baseline |
| Consistent across intents | Lowest-scoring intents | Identified and explained |
| Fast enough for a chatbot | Prediction time for one message on a CPU | Under 200 ms |
| Handles off-topic messages | Off-topic messages flagged, and banking messages wrongly flagged | At least 80% flagged, with under 10% wrongly flagged |
| Usable by a person | Working demo | A typed message returns the top intents with confidence scores |

## 2. Setup

Select a GPU first: **Runtime > Change runtime type > T4 GPU**. Then run these cells in order.

### 2.1 Install libraries

In [ ]:
%pip install -q transformers==5.21.0 datasets==4.8.5 evaluate==0.4.6 accelerate==1.15.0 scikit-learn==1.6.1 gradio==6.30.0

### 2.2 Record library versions

These versions are pinned in `requirements.txt` so that the notebook can be reproduced.

In [ ]:
import platform
from importlib.metadata import version

PACKAGES = [
    "transformers", "datasets", "evaluate", "accelerate", "torch",
    "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "gradio",
]

print(f"# Python {platform.python_version()}")
for package in PACKAGES:
    print(f"{package}=={version(package)}")

### 2.3 Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Select Runtime > Change runtime type > T4 GPU, then run the notebook again."
    )

DEVICE = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 2.4 Connect Google Drive

Trained models are saved to Drive so that they survive Colab disconnects.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/banking-intent-classifier")
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

for directory in (MODELS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

test_file = PROJECT_DIR / "write_test.txt"
test_file.write_text("ok")
assert test_file.read_text() == "ok"
test_file.unlink()

print(f"Drive is ready: {PROJECT_DIR}")

### 2.5 Set the random seed

A fixed seed makes results repeatable.

In [ ]:
import random

import numpy as np

SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Seed: {SEED}")